### Week 6 — Feature Engineering
#### Task
- Create 3 new features
- Retrain model
- Compare results before and after
#### Deliverables
- Feature explanation
- Performance improvement analysis

### 1.Importing Libraries

In [39]:
import pandas as pd
import matplotlib.pyplot as plt

### 2.Load The Dataset

In [40]:
df = pd.read_csv('Titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 3.Creating 3 New Features

In [41]:
# Feature 1: Family 
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature 2: Is Alone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Feature 3: Title

df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
df['Title'] = df['Title'].replace(
['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
'Rare'
)
df['Title'] = df['Title'].replace({
'Mlle':'Miss',
'Ms':'Miss',
'Mme':'Mrs'
})

### 4.Data Preprocessing

In [42]:
### Filling missng values with median
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])

# Encode Categorical variables
from sklearn.preprocessing import LabelEncoder
Le = LabelEncoder()
df['Sex'] = Le.fit_transform(df['Sex'])
df['Title'] = Le.fit_transform(df['Title'])

df = pd.get_dummies(df,columns=['Embarked'],drop_first=True)

# Dropping the column that not used in model training
df = df.drop(columns=['Name','Cabin','Ticket','PassengerId'])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Title,Embarked_Q,Embarked_S
0,0,3,1,22.0,1,0,7.2500,2,0,2,False,True
1,1,1,0,38.0,1,0,71.2833,2,0,3,False,False
2,1,3,0,26.0,0,0,7.9250,1,1,1,False,True
3,1,1,0,35.0,1,0,53.1000,2,0,3,False,True
4,0,3,1,35.0,0,0,8.0500,1,1,2,False,True


### 5.Split the Data Feature And Target

In [44]:
X2 = df.drop('Survived',axis=1)
y2 = df['Survived']

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X2_train, X2_test, y2_train, y2_test = train_test_split(X2,y2,test_size=0.2,random_state=42,stratify=y2)

scaler2 = StandardScaler()
X2_trained_scaled = scaler2.fit_transform(X2_train)
X2_test_scaled = scaler2.fit_transform(X2_test)

print('Train Size',X2_train.shape,'Test Size', X2_test.shape)

Train Size (712, 11) Test Size (179, 11)


### Model 1: Random Forest

In [45]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
Rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=50
)
Rf_model.fit(X2_train,y2_train)

Rf_pred = Rf_model.predict(X2_test)

Rf_result = {
    'Accuracy':accuracy_score(y2_test,Rf_pred),
    'Precision':precision_score(y2_test,Rf_pred),
    'Recall':recall_score(y2_test,Rf_pred)
}

### Model2: SVM(Support Vector Machine)

In [46]:
from sklearn.svm import SVC
svm = SVC(kernel='rbf',random_state=50)
svm.fit(X2_trained_scaled,y2_train)
svm_pred = svm.predict(X2_test_scaled)

Svm_result = {
    'Accuracy':accuracy_score(y2_test,svm_pred),
    'Precision':precision_score(y2_test,svm_pred),
    'Recall':recall_score(y2_test,svm_pred),
}

### Model 3: KNN(K-Nearest Neighbour)

In [47]:
from sklearn.neighbors import KNeighborsClassifier

Knn = KNeighborsClassifier(n_neighbors=5)
Knn.fit(X2_trained_scaled,y2_train)
knn_pred = Knn.predict(X2_test_scaled)

Knn_result = {
    'Accuracy':accuracy_score(y2_test,knn_pred),
    'Precision':precision_score(y2_test,knn_pred),
    'Recall':recall_score(y2_test,knn_pred)
}


### Comparison of Each Model

In [48]:
comparison_df = pd.DataFrame({
    'Random Forest':Rf_result,
    'SVM':Svm_result,
    'KNN':Knn_result
}).T.round(2)

comparison_df

,Accuracy,Precision,Recall
Random Forest,0.83,0.80,0.74
SVM,0.84,0.84,0.74
KNN,0.80,0.76,0.72
